In [ ]:
import cv2
from pathlib import Path

VIDEO_PATH = Path("../../data/lesson_10/traffic.mp4")
CASCADE_PATH = Path("../../data/lesson_10/cars.xml")
RESULTS_DIR = Path("../../data/lesson_10/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REDETECT_FRAMES_PERIOD = 10
FRAME_LIMIT = 15
COLOR_GREEN = (0, 255, 0)
COLOR_RED = (0, 0, 255)

def detect_largest_car(frame, cascade):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cars = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=6, minSize=(30, 30))
  
    if len(cars) == 0:
        return None

    print(f"Detected {len(cars)} cars in the current frame.")
    largest_car = max(cars, key=lambda rect: rect[2] * rect[3])
    return tuple(largest_car)

# Main function to run the tracking (tracker_type can be "KCF" or "CSRT")
def main(tracker_type: str = "KCF"):
    if tracker_type not in ["KCF", "CSRT"]:
        raise ValueError("Invalid tracker_type. Must be 'KCF' or 'CSRT'.")

    video_capture = cv2.VideoCapture(str(VIDEO_PATH))
    cascade = cv2.CascadeClassifier(str(CASCADE_PATH))

    if not video_capture.isOpened():
        raise RuntimeError("Error opening video stream or file")
    if cascade.empty():
        raise RuntimeError("Error loading cars.xml cascade")

    # Calculate delay based on video FPS to maintain real-time playback
    fps = video_capture.get(cv2.CAP_PROP_FPS)
    # Use a default frame duration of 1/30 seconds if FPS is not available or invalid
    frame_duration = 1.0 / fps if fps and fps > 0 else 1.0 / 30.0
    # Ensure a minimum delay of 1 ms to avoid issues with very high FPS videos
    delay_ms = max(1, int(frame_duration * 1000))

    ret, first_frame = video_capture.read()
    if not ret:
        raise RuntimeError("Failed to read first frame")

    # Detect the largest car in the first frame to initialize the tracker
    first_bbox = detect_largest_car(first_frame, cascade)
    if first_bbox is None:
        raise RuntimeError("No cars detected in the first frame")

    # Initialize the tracker with the first frame and the detected bounding box
    if tracker_type == "KCF":
        tracker = cv2.TrackerKCF_create()
    elif tracker_type == "CSRT":
        tracker = cv2.TrackerCSRT_create()
    tracker.init(first_frame, first_bbox)

    frame_index = 0
    detected_count = 0
    redetected_count = 0

    while frame_index < FRAME_LIMIT:
        ret, frame = video_capture.read()
        if not ret:
            print("End of video or cannot read the frame.")
            break

        success, bbox = tracker.update(frame)

        # Redetect if tracking failed or every REDETECT_FRAMES_PERIOD frames
        if not success or (frame_index % REDETECT_FRAMES_PERIOD == 0):
            redetect_bbox = detect_largest_car(frame, cascade)
            if redetect_bbox is not None:
                if tracker_type == "KCF":
                    tracker = cv2.TrackerKCF_create()
                elif tracker_type == "CSRT":
                    tracker = cv2.TrackerCSRT_create()
                tracker.init(frame, redetect_bbox)
                bbox = redetect_bbox
                success = True
                redetected_count += 1

        # Draw bounding box and info on the frame
        if success:
            x, y, w, h = map(int, bbox)
            cv2.rectangle(frame, (x, y), (x + w, y + h), COLOR_GREEN, 2)
            cv2.putText(frame, f"Frame: {frame_index}", (10, 30), cv2.FONT_HERSHEY_DUPLEX, 0.7, COLOR_GREEN, 2)
            detected_count += 1
        else:
            cv2.putText(frame, f"Frame: {frame_index} - Tracking failure", (10, 30), cv2.FONT_HERSHEY_DUPLEX, 0.7, COLOR_RED, 2)

        cv2.putText(frame, f"Detections: {detected_count}", (10, 60), cv2.FONT_HERSHEY_DUPLEX, 0.7, COLOR_GREEN, 2)
        cv2.putText(frame, f"Redetections: {redetected_count}", (10, 90), cv2.FONT_HERSHEY_DUPLEX, 0.7, COLOR_GREEN, 2)
        tracker_type_path = tracker_type.lower()
        tracker_dir = RESULTS_DIR / tracker_type_path
        tracker_dir.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(tracker_dir / f"output_frame_{frame_index}.jpg"), frame)  # Save the frame for debugging
        cv2.imshow("Car Tracking", frame)

        if cv2.waitKey(delay_ms) & 0xFF == ord("q"):
            break

        frame_index += 1

    video_capture.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

if __name__ == "__main__":
    main(tracker_type="CSRT")  # Change to "CSRT" to use the CSRT tracker

Detected 4 cars in the current frame.
Detected 6 cars in the current frame.
Detected 4 cars in the current frame.


: 

Do you see any differences? If so, what are they?
> Да, в плане детекции, KCF кажется менее точным,чем CSRT, квадраты иногда смещаются, но он кажется более быстрым. CSRT рисует более точные квадраты, то есть более стабильно трекает объект, но из-за этого думаю медленнее. То есть как всегда, палка о двух концах, нужно подбирать под конкретную задачу и искать сбалансированый вариант путем компромисса между скоростю и точностью 

Does one tracker perform better than the other? In what way?
> Как уже написал выше, CSRT хорош когда нужна точность и стабильность, а KCF когда нужна скорость в случае с слабым железом или точность не так критична. Я бы предпочел CSRT так как он кажется более универсальным и стабильным вариантом. Естественно если еще откалибровать minNeighbors и scaleFactor для детектора.